# 01 · Carga de datos → Delta
### Prueba final MLOps · Cesar Romero

**Objetivos**
- Cargar los datos necesarios para el entrenamiento.
- Persistir los datos usando **tablas Delta**.

**Dataset:** `load_breast_cancer` de scikit-learn (no requiere descargas externas).
`target = 1` → benigno · `target = 0` → maligno.

**Salida de este notebook:** tabla Delta `<catalog>.mlops_final.cancer_raw`

> Adjunta el notebook a compute **Serverless**.

In [0]:
print("01. Carga de datos")

## 1. Configuración

Usamos `current_catalog()` en lugar de escribir `workspace` a mano: así el notebook
funciona igual si el workspace tiene otro catálogo por defecto.
Estas mismas 4 variables se repiten en los 3 notebooks — es el contrato entre etapas.

In [0]:
# Configuración central (idéntica en los 3 notebooks)
CATALOG = spark.sql("SELECT current_catalog()").collect()[0][0]
SCHEMA  = "mlops_final"

RAW_TABLE      = f"{CATALOG}.{SCHEMA}.cancer_raw"
FEATURE_TABLE  = f"{CATALOG}.{SCHEMA}.cancer_features"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

print("Catálogo :", CATALOG)
print("Esquema  :", SCHEMA)
print("Raw      :", RAW_TABLE)
print("Features :", FEATURE_TABLE)

## 2. Carga del dataset

Nos quedamos con 6 variables `mean *` y las renombramos a nombres cortos.
Motivo: el endpoint REST del paso 5 recibirá exactamente estas columnas, y los
espacios en los nombres (`mean radius`) complican el JSON del payload.

`patient_id` es la **clave de entidad**: sin ella no existe Feature Store.

In [0]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame

COLS = {
    "mean radius":      "radius",
    "mean texture":     "texture",
    "mean perimeter":   "perimeter",
    "mean area":        "area",
    "mean smoothness":  "smoothness",
    "mean compactness": "compactness",
}

pdf = df[list(COLS.keys()) + ["target"]].rename(columns=COLS)
pdf.insert(0, "patient_id", range(1, len(pdf) + 1))   # clave de entidad

print("Filas:", len(pdf), "| Columnas:", list(pdf.columns))
print("\nDistribución de clases:")
print(pdf["target"].value_counts().rename({0: "maligno (0)", 1: "benigno (1)"}))
display(pdf.head(10))

## 3. Persistencia en Delta

`mode("overwrite")` + `overwriteSchema` hace la celda **idempotente**: el Job del paso 6
puede reejecutarse cuantas veces quiera sin duplicar filas ni romperse si cambia el esquema.

Esto es lo que se evalúa: los datos quedan en una tabla Delta gobernada por Unity Catalog,
no en un CSV suelto.

In [0]:
sdf = spark.createDataFrame(pdf)

(sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RAW_TABLE))

print(f"✅ Tabla Delta creada/actualizada: {RAW_TABLE}")
display(spark.table(RAW_TABLE).limit(10))

## 4. Verificación (evidencia)

`DESCRIBE HISTORY` sólo funciona sobre tablas Delta → es la prueba de que la
persistencia cumple el requisito. Además, la **versión** que devuelve la usaremos
en el notebook 03 como parámetro de auditoría en MLflow.

In [0]:
print("Formato y ubicación:")
display(spark.sql(f"DESCRIBE DETAIL {RAW_TABLE}").select("format", "numFiles", "sizeInBytes", "location"))

print("Historial Delta (time travel disponible):")
display(spark.sql(f"DESCRIBE HISTORY {RAW_TABLE}").select("version", "timestamp", "operation"))

---
### Cierre notebook 01

```
scikit-learn  →  pandas  →  Spark DataFrame  →  Delta: cancer_raw
```

**Evidencia para el examen:** screenshot de Catalog Explorer mostrando `cancer_raw`
dentro de `mlops_final`, o el output de `DESCRIBE DETAIL` con `format = delta`.

Continúa con el notebook **02**.